# Module 08 — Notebook 2: Distributions and Spread

## Learning Objectives

By the end of this notebook, you will be able to:

- Compute variance and standard deviation using `statistics` stdlib and manual list math
- Compute range and interquartile range (IQR) from a list of scores
- Explain why two models can have the same mean but very different reliability
- Interpret what a high vs. low standard deviation means for evaluation results

**Estimated time:** ~20 minutes

## Why This Matters for AI Research Engineering

Two models can both score 0.82 on average. But one might score between 0.79 and 0.85 on every task (consistent and predictable), while the other swings between 0.50 and 0.99 (wildly unreliable). The mean alone won't tell you this.

**Spread** — variance, standard deviation, range, IQR — tells you how reliable a model is. In safety contexts, high spread is a red flag: it means the model's behavior is hard to predict. A model that sometimes scores 0.99 and sometimes scores 0.40 might be more dangerous than one that consistently scores 0.75.

Understanding spread is also the foundation for everything in Notebook 3: t-tests and confidence intervals only make sense if you understand standard deviation first.

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length
import statistics
print("Setup complete.")

## 1. Variance

**Variance** measures how spread out values are from the mean. The formula:

```
variance = sum((x - mean)^2 for x in data) / (n - 1)
```

We divide by `n - 1` (not `n`) when working with a **sample** rather than the full population. This is called Bessel's correction and gives a less biased estimate. Python's `statistics.variance()` uses `n - 1` by default.

Variance is in squared units — if your scores are in [0, 1], variance is in [0, 1]². That's why we usually prefer standard deviation.

In [ ]:
import statistics

# Two models, same mean — very different spread
consistent_model = [0.80, 0.81, 0.79, 0.82, 0.80, 0.81, 0.79, 0.80]
erratic_model    = [0.50, 0.95, 0.60, 0.99, 0.45, 0.92, 0.55, 0.98]

print("=== Consistent model ===")
print(f"  Mean:     {statistics.mean(consistent_model):.4f}")
print(f"  Variance: {statistics.variance(consistent_model):.6f}")

print("\n=== Erratic model ===")
print(f"  Mean:     {statistics.mean(erratic_model):.4f}")
print(f"  Variance: {statistics.variance(erratic_model):.6f}")

print("\nSame mean, wildly different variance!")

In [ ]:
# Manual variance calculation — useful to understand what's happening inside
data = [0.80, 0.81, 0.79, 0.82, 0.80, 0.81, 0.79, 0.80]

mean = statistics.mean(data)
n = len(data)

# Step 1: squared deviations from the mean
squared_devs = [(x - mean) ** 2 for x in data]
print("Squared deviations:", [round(d, 8) for d in squared_devs])

# Step 2: sum and divide by n-1
variance_manual = sum(squared_devs) / (n - 1)
variance_stdlib = statistics.variance(data)

print(f"\nManual variance: {variance_manual:.8f}")
print(f"Stdlib variance: {float(variance_stdlib):.8f}")
print(f"Match: {abs(variance_manual - float(variance_stdlib)) < 1e-10}")

## 2. Standard Deviation

**Standard deviation** is just the square root of variance — it brings the units back to the same scale as your original data.

```
std_dev = sqrt(variance)
```

A rule of thumb for normally distributed data: about 68% of values fall within 1 standard deviation of the mean, and 95% within 2 standard deviations.

In practice: if your eval scores have a std dev of 0.03, scores are tightly clustered (reliable model). If std dev is 0.20, scores are all over the place (unreliable model).

In [ ]:
import statistics
import math

consistent_model = [0.80, 0.81, 0.79, 0.82, 0.80, 0.81, 0.79, 0.80]
erratic_model    = [0.50, 0.95, 0.60, 0.99, 0.45, 0.92, 0.55, 0.98]

# statistics.stdev() is sqrt(variance)
for name, model in [("Consistent", consistent_model), ("Erratic", erratic_model)]:
    mean = statistics.mean(model)
    std  = statistics.stdev(model)
    lo   = mean - std
    hi   = mean + std
    print(f"{name} model:")
    print(f"  mean={mean:.4f}  std={std:.4f}")
    print(f"  Typical range: [{lo:.4f}, {hi:.4f}]")
    print()

## 3. Range and IQR

**Range** = max − min. Simple but very sensitive to outliers.

**Interquartile Range (IQR)** = 75th percentile − 25th percentile. This is the spread of the middle 50% of your data, and it's much more robust to outliers than range.

The `statistics` stdlib has `statistics.quantiles(data, n=4)` which returns the three quartile boundaries (Q1, Q2, Q3). So `IQR = Q3 - Q1`.

> **JS analogy:** Think of IQR like the interquartile equivalent of `Array.filter()` — you're ignoring the top 25% and bottom 25% and looking at what the bulk of your data is doing.

In [ ]:
import statistics

# Scores including two outliers
scores_with_outliers = [0.82, 0.85, 0.79, 0.83, 0.81, 0.84, 0.80, 0.82, 0.05, 0.99]

# Range — sensitive to outliers
score_range = max(scores_with_outliers) - min(scores_with_outliers)
print(f"Range: {score_range:.4f}  (influenced by 0.05 and 0.99)")

# IQR — robust to outliers
# statistics.quantiles returns [Q1, Q2, Q3] by default with n=4
q1, q2, q3 = statistics.quantiles(scores_with_outliers, n=4)
iqr = q3 - q1
print(f"Q1={q1:.4f}  Q2={q2:.4f}  Q3={q3:.4f}")
print(f"IQR: {iqr:.4f}  (reflects where the bulk of scores sit)")

## 4. Interpreting Spread in Context

Here's a quick mental model for interpreting standard deviation on eval scores scaled 0–1:

| Std Dev | Interpretation |
|---------|----------------|
| < 0.05 | Very consistent — model behaves predictably |
| 0.05–0.15 | Moderate variation — some task-to-task difference |
| > 0.15 | High variation — model is unreliable across tasks |

These thresholds are heuristics, not hard rules. Context matters: a safety classifier with std dev 0.20 on refusal rates is much more alarming than the same spread on creative writing scores.

In [ ]:
import statistics

def reliability_label(scores):
    """Classify a model's reliability based on its score standard deviation."""
    std = statistics.stdev(scores)
    if std < 0.05:
        return f"Very consistent (std={std:.4f})"
    elif std < 0.15:
        return f"Moderate variation (std={std:.4f})"
    else:
        return f"High variation — unreliable (std={std:.4f})"

model_alpha = [0.81, 0.82, 0.80, 0.83, 0.81, 0.82]
model_beta  = [0.65, 0.88, 0.72, 0.90, 0.58, 0.85]
model_gamma = [0.40, 0.92, 0.30, 0.95, 0.25, 0.98]

print(f"Model Alpha: {reliability_label(model_alpha)}")
print(f"Model Beta:  {reliability_label(model_beta)}")
print(f"Model Gamma: {reliability_label(model_gamma)}")

## Exercise 1 — Compute Variance and Standard Deviation

Given the scores below, compute the variance and standard deviation using `statistics.variance()` and `statistics.stdev()`. Round both to 6 decimal places and store in `score_variance` and `score_stdev`.

In [ ]:
import statistics

task_scores = [0.72, 0.85, 0.68, 0.91, 0.74, 0.88, 0.65, 0.79, 0.83, 0.71]

# YOUR CODE HERE
score_variance = None  # float, rounded to 6 decimal places
score_stdev = None     # float, rounded to 6 decimal places

In [ ]:
check_type(score_variance, float, "score_variance is a float")
check_type(score_stdev, float, "score_stdev is a float")
check_approx(score_variance, 0.006423, 0.0001, "score_variance is correct")
check_approx(score_stdev, 0.080143, 0.001, "score_stdev is correct")

## Exercise 2 — Compute IQR

Compute the interquartile range (IQR) of the same `task_scores` list. Use `statistics.quantiles()` with `n=4`. Store the result in `score_iqr`, rounded to 4 decimal places.

In [ ]:
import statistics

task_scores = [0.72, 0.85, 0.68, 0.91, 0.74, 0.88, 0.65, 0.79, 0.83, 0.71]

# YOUR CODE HERE
score_iqr = None  # float, rounded to 4 decimal places

In [ ]:
check_type(score_iqr, float, "score_iqr is a float")
check_approx(score_iqr, 0.1475, 0.01, "score_iqr is correct")

## Exercise 3 — Manual Variance

Compute variance **without** using `statistics.variance()`. Use a list comprehension to compute squared deviations from the mean, then divide the sum by `n - 1`. Store in `variance_manual`, rounded to 6 decimal places.

Then verify it matches `statistics.variance()` (within floating-point tolerance).

In [ ]:
import statistics

data = [0.72, 0.85, 0.68, 0.91, 0.74, 0.88, 0.65, 0.79, 0.83, 0.71]

# YOUR CODE HERE — do NOT use statistics.variance()
variance_manual = None  # float, rounded to 6 decimal places

In [ ]:
check_type(variance_manual, float, "variance_manual is a float")
check_approx(variance_manual, float(statistics.variance(data)), 0.0001, "manual variance matches statistics.variance()")

## Exercise 4 — Reliability Comparison

Two models were evaluated on 8 tasks. Compare their reliability. Compute the standard deviation of each, determine which is more reliable (lower std dev), and store the name of the more reliable model in `more_reliable` as either `"model_x"` or `"model_y"`.

In [ ]:
import statistics

model_x = [0.78, 0.80, 0.77, 0.81, 0.79, 0.80, 0.78, 0.79]
model_y = [0.55, 0.93, 0.62, 0.88, 0.49, 0.97, 0.58, 0.90]

# YOUR CODE HERE
std_x = None          # float
std_y = None          # float
more_reliable = None  # "model_x" or "model_y"

In [ ]:
check_type(std_x, float, "std_x is a float")
check_type(std_y, float, "std_y is a float")
check_equal(more_reliable, "model_x", "model_x is more reliable (lower std dev)")

## Wrap-Up

| Measure | Python | Robust to outliers? | What it tells you |
|---------|--------|---------------------|-------------------|
| Variance | `statistics.variance(data)` | No | Average squared distance from mean |
| Std deviation | `statistics.stdev(data)` | No | Same scale as data; "typical" distance from mean |
| Range | `max(data) - min(data)` | No | Total spread; dominated by extremes |
| IQR | `Q3 - Q1` via `statistics.quantiles(data, n=4)` | Yes | Spread of middle 50%; ignores outliers |

**Key insight:** Mean tells you where a model performs. Std dev tells you how consistently. In safety-critical settings, consistent underperformance may be better than unpredictable high variance.

**Next:** Notebook 3 — Comparing Groups: t-tests, effect size (Cohen's d), and bootstrap confidence intervals.